# nanochat.cpp end to end on Kaggle (1x T4)

Build the CUDA library for the T4 (`sm_75`), prepare ClimbMix and the
`NCTOKEN1` tokenizer, pretrain a small base model, and post-train it with
supervised fine-tuning in one process.

The runtime uses one T4. The C++ workflow has no distributed mode
(`docs/host-portability.md`). The Python surface is `docs/python.md`. The
post-training path mirrors `docs/post-training.md`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

# The Hugging Face token lets the data download run without a prompt.
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret; the download may be rate limited")

REPO = Path("/kaggle/working/nanochat.cpp")
BASE = Path("/kaggle/working/nanochat-cache")
HOME = Path(os.environ["HOME"])

# The host contract lives in tools/kaggle/bootstrap.sh. It writes the rest
# to ~/.nanochat.env, and the next cell reads it.
os.environ["PATH"] = os.pathsep.join([
    str(HOME / ".local/bin"),
    str(HOME / ".local/node/bin"),
    "/usr/local/cuda/bin",
    os.environ["PATH"],
])

print("environment ready:", REPO)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total",
                      "--format=csv,noheader"], text=True,
                     capture_output=True).stdout.strip())


## Build

Clone the repository, prepare the Kaggle session, and build the shared library
for the T4.


In [ ]:
if not REPO.is_dir():
    subprocess.run(["git", "clone",
                    "https://github.com/ming6ao/nanochat.cpp", str(REPO)],
                   check=True)
subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
subprocess.run(["tools/kaggle/bootstrap.sh"], cwd=str(REPO), check=True)

# Read the sandbox backend and the CUDA path from the host contract.
for line in (HOME / ".nanochat.env").read_text().splitlines():
    if line.startswith("export ") and "=" in line:
        key, _, value = line[len("export "):].partition("=")
        if key in ("NANOCHAT_SANDBOX_BACKEND", "CUDA_HOME"):
            os.environ[key] = value.strip().strip('"')

revision = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short",
                           "HEAD"], text=True, capture_output=True)
print("revision:", revision.stdout.strip())

sys.path.insert(0, str(REPO / "python"))

import nanochat_cpp as nc

# Build the shared library for the T4 and pin the path before the first load.
os.environ["NANOCHAT_CPP_LIB"] = str(nc.build(
    backend="cuda", precision="fp16", arch="sm_75"))
print("CUDA library:", os.environ["NANOCHAT_CPP_LIB"])


## Data and tokenizer

One ClimbMix shard trains the tokenizer and the base model. The validation
shard is the reference split.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "huggingface_hub"], check=True)

from huggingface_hub import snapshot_download

DATA = BASE / "base_data_climbmix"
TRAIN_SHARD = "shard_00000.parquet"   # one shard is enough for a small model
VAL_SHARD = "shard_06542.parquet"     # the reference validation split

if not (DATA / VAL_SHARD).is_file():
    snapshot_download(repo_id="karpathy/climbmix-400b-shuffle",
                      repo_type="dataset", local_dir=str(DATA),
                      allow_patterns=[TRAIN_SHARD, VAL_SHARD],
                      token=os.environ.get("HF_TOKEN"))

print("train:", (DATA / TRAIN_SHARD).name,
      "validation:", (DATA / VAL_SHARD).name)


In [ ]:
artifact = BASE / "tokenizer" / "tokenizer.nctoken"

if not artifact.is_file():
    artifact.parent.mkdir(parents=True, exist_ok=True)
    # The orchestration layer calls tools/nanochat for the build and the run.
    nc.toolchain.build(["//src/tokenizer:tok_train_main"])
    nc.toolchain.run("t0-cpu", [
        "bazel-bin/src/tokenizer/tok_train_main",
        "--parquet", f"{DATA}/{TRAIN_SHARD}",
        "--vocab-size", "32768",
        "--out", str(artifact),
    ])

tokenizer = nc.Tokenizer.load(str(artifact))
assert tokenizer.decode(tokenizer.encode("The capital of France is")) == \
    "The capital of France is", "the tokenizer round trip failed"
print("tokenizer:", tokenizer.vocab_size, "tokens")


## Pretrain

The plan sets the model shape, the horizon, and the scaled rates. One optimizer
step consumes one micro batch.


In [ ]:
import time

plan = nc.plan.compute_plan(
    depth=8, vocab_size=tokenizer.vocab_size, seq_len=512,
    device_batch_size=8, total_batch_size=4096, num_iterations=50)
model = nc.Model(plan.config, device="cuda", seed=42)
print("device:", model.device, "scaling params:", f"{plan.scaling_params:,}")

train_data = nc.TokenData(
    parquet=f"{DATA}/{TRAIN_SHARD}", tokenizer=tokenizer,
    seq_len=plan.config.seq_len, batch=plan.device_batch_size, threads=4)
optimizer = nc.Optimizer(model, num_iterations=plan.num_iterations,
                         **plan.optimizer, **plan.scheduler)
trainer = nc.Trainer(model, train_data, num_iterations=plan.num_iterations,
                     optimizer=optimizer)

# One optimizer step per iteration, so a step consumes one micro batch.
tokens_per_step = plan.device_batch_size * plan.config.seq_len
history = {"step": [], "loss": [], "tokens_per_sec": []}

previous = time.perf_counter()
for step, loss in trainer:
    now = time.perf_counter()
    history["step"].append(step)
    history["loss"].append(loss)
    history["tokens_per_sec"].append(tokens_per_step / (now - previous))
    previous = now
    if step == 1 or step % 10 == 0 or step == plan.num_iterations:
        print(f"step {step:3d}/{plan.num_iterations}  loss {loss:.4f}")

checkpoint = Path("/kaggle/working/base.nchkpt01")
model.save(str(checkpoint))

base_bpb = nc.evaluate(
    model, parquet=str(DATA / VAL_SHARD), tokenizer=tokenizer,
    tokens=plan.device_batch_size * plan.config.seq_len * 4,
    batch=plan.device_batch_size, seed=43).bpb
print(f"base checkpoint: {checkpoint.stat().st_size:,} bytes, "
      f"{base_bpb:.4f} bits per byte")

assert history["loss"][-1] < history["loss"][0], "the pretraining loss decreased"


## Pretraining metrics

The loss and the throughput from the run above.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

frame = pd.DataFrame(history).set_index("step")
axes = frame[["loss", "tokens_per_sec"]].plot(
    subplots=True, figsize=(11, 5), grid=True,
    title=["Pretraining loss", "Throughput"])
axes[0].set_ylabel("cross-entropy loss")
axes[1].set_ylabel("tokens / second")
axes[0].figure.suptitle(f"nanochat.cpp pretraining on {model.device}")
plot_path = Path("/kaggle/working/pretraining_metrics.png")
axes[0].figure.savefig(plot_path, dpi=120)
print("saved:", plot_path)
plt.show()


## Post-train: supervised fine-tuning

The base model becomes a chat model on a small conversation set.
`nc.render_conversation` returns the token ids and a loss mask. `mask = 1` marks
the assistant text, the assistant end token, and the python tool-call tokens.
`mask = 0` marks the BOS, the user text, and the tool outputs. The packer shifts
the mask by one position, because position `j` predicts token `j + 1`. A masked
target becomes `-1`, and the loss divides by the valid tokens only.


In [ ]:
def shifted_targets(ids, mask, bos):
    """Shift by one and set a masked target to `-1` (loss ignore index)."""
    targets = ids[1:] + [bos]
    keep = mask[1:] + [0]
    return [token if flag else -1 for token, flag in zip(targets, keep)]


def pack_conversations(conversations, tokenizer, seq_len):
    """Pack conversations into padded `(tokens, targets)` rows.

    `nc.render_conversation` returns the ids and the loss mask. This helper
    pads the row and shifts the mask onto the targets.
    """
    bos = tokenizer.get_bos_token_id()
    rows, ids, mask = [], [], []

    def flush():
        if not ids:
            return
        pad = seq_len - len(ids)
        full_ids = ids + [bos] * pad
        full_mask = mask + [0] * pad
        rows.append((full_ids, shifted_targets(full_ids, full_mask, bos)))

    for conversation in conversations:
        part_ids, part_mask = nc.render_conversation(conversation, tokenizer)
        if len(ids) + len(part_ids) > seq_len:
            flush()
            ids, mask = [], []
        ids.extend(part_ids)
        mask.extend(part_mask)
    flush()
    return rows


FACTS = [("France", "Paris"), ("Japan", "Tokyo"), ("Italy", "Rome"),
         ("Spain", "Madrid"), ("Egypt", "Cairo"), ("Canada", "Ottawa"),
         ("Brazil", "Brasilia"), ("Kenya", "Nairobi")]
CONVERSATIONS = [
    {"messages": [
        {"role": "user", "content": f"What is the capital of {country}?"},
        {"role": "assistant", "content": f"The capital of {country} is {capital}."}]}
    for country, capital in FACTS]
CONVERSATIONS += [
    {"messages": [
        {"role": "user", "content": f"What is {a} + {b}?"},
        {"role": "assistant", "content": f"{a} + {b} = {a + b}."}]}
    for a, b in [(2, 3), (5, 7), (8, 1), (4, 6), (9, 2), (3, 9), (7, 7), (6, 5),
                 (1, 8), (5, 5), (2, 9), (4, 4)]]


def complete(model, tokenizer, prompt, max_tokens=12):
    ids = tokenizer.encode(prompt)
    row = model.generate(ids, max_tokens=max_tokens, temperature=0.0)[0]
    return tokenizer.decode(
        [token for token, keep in zip(row.tokens, row.mask) if keep])


In [ ]:
SFT_SEQ = 128
SFT_STEPS = 40

# Demo fine-tuning rates. They mirror the reference SFT starting point.
sft_optimizer = nc.Optimizer(
    model,
    embedding_lr=0.3, unembedding_lr=0.008, matrix_lr=0.02, scalar_lr=0.5,
    weight_decay=0.0,
    num_iterations=SFT_STEPS, warmup_steps=2, warmdown_ratio=0.5,
    final_lr_frac=0.0)

sft_rows = pack_conversations(CONVERSATIONS, tokenizer, SFT_SEQ)
print(f"{len(sft_rows)} packed rows of {SFT_SEQ} tokens")

base_completion = complete(model, tokenizer, "The capital of France is")

sft_losses = []
for step in range(1, SFT_STEPS + 1):
    tokens, targets = sft_rows[(step - 1) % len(sft_rows)]
    sft_optimizer.zero_grad()
    loss = model.forward_loss(tokens, targets, batch=1, seq=SFT_SEQ)
    model.backward()
    sft_optimizer.step(step)
    sft_losses.append(loss)
    if step == 1 or step % 10 == 0 or step == SFT_STEPS:
        print(f"sft step {step:3d}/{SFT_STEPS}  loss {loss:.4f}")

sft_completion = complete(model, tokenizer, "The capital of France is")

sft_checkpoint = Path("/kaggle/working/chatsft.nchkpt01")
model.save(str(sft_checkpoint))
print("base completion:", base_completion)
print("sft completion: ", sft_completion)
print(f"chatsft checkpoint: {sft_checkpoint.stat().st_size:,} bytes")
early = sum(sft_losses[:5]) / 5
late = sum(sft_losses[-5:]) / 5
assert late < early, "the fine-tuning loss decreased"


## Final checks

Reload the post-trained checkpoint and score it.


In [ ]:
reloaded = nc.Model(plan.config, device="cuda", seed=0)
reloaded.load(str(sft_checkpoint))
batch_tokens = list(range(plan.config.seq_len))
loss = reloaded.forward_loss(batch_tokens, batch_tokens[1:] + [0],
                             batch=1, seq=plan.config.seq_len)
assert loss == loss, "the reloaded model scores a finite loss"

sft_bpb = nc.evaluate(
    model, parquet=str(DATA / VAL_SHARD), tokenizer=tokenizer,
    tokens=plan.device_batch_size * plan.config.seq_len * 4,
    batch=plan.device_batch_size, seed=43).bpb
print(f"base {base_bpb:.4f} bits per byte, post-trained {sft_bpb:.4f}")
print("final checks passed")
